In [1]:
import copy
import random
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms

In [13]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [15]:
def get_mnist_base():
    """Downloads MNIST and normalizes pixel values."""
    transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
    )
    train_dataset = datasets.MNIST(
        root="./data", train=True, download=True, transform=transform
    )
    test_dataset = datasets.MNIST(
        root="./data", train=False, download=True, transform=transform
    )
    return train_dataset, test_dataset

In [16]:
def build_custom_dataloader(
    base_dataset,
    num_images=20000,
    noise_rate=0.0,
    imbalance_ratio=1.0,
    batch_size=128,
    seed=42,
):
    """Subsamples MNIST to exact N, applies configurable majority:minority class

    imbalance across half the classes, and injects symmetric label noise.
    """
    rng = np.random.default_rng(seed)
    targets = np.array(base_dataset.targets)
    selected_indices = []

    if imbalance_ratio <= 1.0:
        per_class = num_images // 10
        diff = num_images - (per_class * 10)
        for cls in range(10):
            cls_idx = np.where(targets == cls)[0]
            count = per_class + (diff if cls == 0 else 0)
            chosen = rng.choice(cls_idx, size=count, replace=False)
            selected_indices.extend(chosen)
    else:
        # Classes 0-4 are majority (ratio * k), Classes 5-9 are minority (k)
        # Total N = 5 * (ratio * k) + 5 * k = 5 * (ratio + 1) * k
        k = num_images / (5.0 * (imbalance_ratio + 1.0))
        minority_count = max(1, int(round(k)))
        majority_count = int(round(imbalance_ratio * k))
        diff = num_images - (5 * majority_count + 5 * minority_count)

        for cls in range(10):
            cls_idx = np.where(targets == cls)[0]
            count = majority_count if cls < 5 else minority_count
            if cls == 0:
                count += diff
            chosen = rng.choice(cls_idx, size=count, replace=False)
            selected_indices.extend(chosen)

    selected_indices = np.array(selected_indices)
    rng.shuffle(selected_indices)

    x_data = torch.stack([base_dataset[i][0] for i in selected_indices])
    y_data = torch.tensor(targets[selected_indices], dtype=torch.long)

    if noise_rate > 0.0:
        num_noisy = int(len(y_data) * noise_rate)
        noisy_indices = rng.choice(len(y_data), size=num_noisy, replace=False)
        for idx in noisy_indices:
            orig_label = y_data[idx].item()
            possible_labels = [c for c in range(10) if c != orig_label]
            y_data[idx] = rng.choice(possible_labels)

    custom_dataset = TensorDataset(x_data, y_data)
    return DataLoader(custom_dataset, batch_size=batch_size, shuffle=True)

In [8]:
class DynamicMLP(nn.Module):

    def __init__(
        self,
        input_dim=784,
        num_classes=10,
        hidden_layers=2,
        nodes_per_layer=256,
        dropout_rate=0.2,
    ):
        super().__init__()
        layers = [nn.Flatten()]

        in_features = input_dim
        for _ in range(hidden_layers):
            layers.append(nn.Linear(in_features, nodes_per_layer))
            layers.append(nn.ReLU())
            if dropout_rate > 0.0:
                layers.append(nn.Dropout(dropout_rate))
            in_features = nodes_per_layer

        layers.append(nn.Linear(in_features, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [18]:
def evaluate_config(train_base, test_loader, config, epochs=10, seed=42):
    set_seed(seed)

    train_loader = build_custom_dataloader(
        train_base,
        num_images=config["num_images"],
        noise_rate=config["noise_rate"],
        imbalance_ratio=config["imbalance_ratio"],
        batch_size=config["batch_size"],
        seed=seed,
    )

    model = DynamicMLP(
        hidden_layers=config["hidden_layers"],
        nodes_per_layer=config["nodes_per_layer"],
        dropout_rate=config["dropout_rate"],
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config["lr"])

    model.train()
    for _ in range(epochs):
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            preds = model(x_batch).argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    return 100.0 * correct / total

In [19]:
def recommend_binary_levels(results, min_valid_acc=65.0):
    """Selects the two candidate levels that maximize accuracy separation

    while filtering out diverged/collapsed runs (< min_valid_acc).
    """
    valid = [(val, acc) for val, acc in results if acc >= min_valid_acc]
    if len(valid) < 2:
        valid = results  # Fallback if all runs are low

    low_pair = min(valid, key=lambda x: x[1])
    high_pair = max(valid, key=lambda x: x[1])
    spread = high_pair[1] - low_pair[1]
    return low_pair, high_pair, spread

In [20]:
if __name__ == "__main__":
    print(f"Screening Candidate Levels on: {device}\n")
    train_base, test_base = get_mnist_base()
    test_loader = DataLoader(test_base, batch_size=512, shuffle=False)

    # Center-point baseline held constant during screening
    center_config = {
        "num_images": 10000,
        "noise_rate": 0.0,
        "imbalance_ratio": 1.0,
        "lr": 1e-3,
        "batch_size": 128,
        "dropout_rate": 0.2,
        "hidden_layers": 2,
        "nodes_per_layer": 128,
    }

    # Candidate grid: test 4-5 levels per factor to observe response curves
    candidate_grid = {
        "num_images": [1000, 2500, 5000, 15000, 30000],
        "noise_rate": [0.0, 0.15, 0.30, 0.45, 0.60],
        "imbalance_ratio": [1.0, 5.0, 10.0, 25.0, 50.0],
        "lr": [1e-4, 5e-4, 1e-3, 5e-3, 2e-2],
        "batch_size": [16, 64, 128, 512, 1024],
        "dropout_rate": [0.0, 0.2, 0.4, 0.6, 0.75],
        "hidden_layers": [1, 2, 3, 5, 7],
        "nodes_per_layer": [16, 32, 64, 256, 512],
    }

    recommended_design = {}

    for param, candidates in candidate_grid.items():
        print(f"--- Screening Factor: {param} ---")
        results = []
        for val in candidates:
            cfg = copy.deepcopy(center_config)
            cfg[param] = val
            acc = evaluate_config(train_base, test_loader, cfg, epochs=10)
            results.append((val, acc))
            print(f"  {param} = {str(val):<8} -> Test Acc: {acc:6.2f}%")

        low_pair, high_pair, spread = recommend_binary_levels(results)
        recommended_design[param] = {
            "-1 (Low Acc)": low_pair[0],
            "+1 (High Acc)": high_pair[0],
            "Expected Spread": round(spread, 2),
        }
        print(
            f"  >> Recommended Pair: {low_pair[0]} ({low_pair[1]:.2f}%) vs. {high_pair[0]} ({high_pair[1]:.2f}%) | Spread: {spread:.2f}%\n"
        )

    # Summary Table for your 2^(k-p) Fractional Factorial Design
    print("=" * 78)
    print("SUGGESTED 2-LEVEL FRACTIONAL FACTORIAL LEVELS (-1 vs +1)")
    print("=" * 78)
    print(
        f"{'Factor':<18} | {'Level -1 (Low)':<16} | {'Level +1 (High)':<16} | {'Single-Factor Spread':<20}"
    )
    print("-" * 78)
    for param, info in recommended_design.items():
        print(
            f"{param:<18} | {str(info['-1 (Low Acc)']):<16} | {str(info['+1 (High Acc)']):<16} | {info['Expected Spread']:>+18.2f}%"
        )

Screening Candidate Levels on: cuda

--- Screening Factor: num_images ---


/tmp/ipykernel_26761/336597054.py:14: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  targets = np.array(base_dataset.targets)


  num_images = 1000     -> Test Acc:  88.47%
  num_images = 2500     -> Test Acc:  92.01%
  num_images = 5000     -> Test Acc:  93.92%
  num_images = 15000    -> Test Acc:  96.38%
  num_images = 30000    -> Test Acc:  96.95%
  >> Recommended Pair: 1000 (88.47%) vs. 30000 (96.95%) | Spread: 8.48%

--- Screening Factor: noise_rate ---
  noise_rate = 0.0      -> Test Acc:  95.55%
  noise_rate = 0.15     -> Test Acc:  94.73%
  noise_rate = 0.3      -> Test Acc:  92.95%
  noise_rate = 0.45     -> Test Acc:  91.39%
  noise_rate = 0.6      -> Test Acc:  88.02%
  >> Recommended Pair: 0.6 (88.02%) vs. 0.0 (95.55%) | Spread: 7.53%

--- Screening Factor: imbalance_ratio ---
  imbalance_ratio = 1.0      -> Test Acc:  95.55%
  imbalance_ratio = 5.0      -> Test Acc:  93.83%
  imbalance_ratio = 10.0     -> Test Acc:  91.35%
  imbalance_ratio = 25.0     -> Test Acc:  87.85%
  imbalance_ratio = 50.0     -> Test Acc:  77.62%
  >> Recommended Pair: 50.0 (77.62%) vs. 1.0 (95.55%) | Spread: 17.93%

--- Sc